## Assertion Commands

### 1. Non associativy in FP

In [1]:
!nvfortran assertions/1_non_aso.cuf
!./a.out

 n:       1000000
 n*(n+1)/2:              500000500000
 from sum(a):           4.9994138E+11
 incr accumulation:     4.9994138E+11
 decr accumulation:     4.9987368E+11
 from sum(a_d):         5.0000003E+11


### 2. Debugging: print & write

In [2]:
!nvfortran assertions/2_debug_simp.cuf
!./a.out

 i, a(i):           33            4
 i, a(i):           31            4
 i, a(i):           32            4
 Program Passed


### 3. Debugging: cuda-gdb

For devices with 6.0 cc

#### a. Compilation

In [ ]:
!nvfortran -g -gpu=nordc assertions/3_step.cuf
!cuda-gdb ./a.out

NVIDIA (R) cuda-gdb 12.8
Portions Copyright (C) 2007-2024 NVIDIA Corporation
Based on GNU gdb 13.2
Copyright (C) 2023 Free Software Foundation, Inc.
License GPLv3+: GNU GPL version 3 or later <http://gnu.org/licenses/gpl.html>
This is free software: you are free to change and redistribute it.
There is NO WARRANTY, to the extent permitted by law.
Type "show copying" and "show warranty" for details.
This CUDA-GDB was configured as "x86_64-pc-linux-gnu".
Type "show configuration" for configuration details.
For bug reporting instructions, please see:
<https://forums.developer.nvidia.com/c/developer-tools/cuda-developer-tools/cuda-gdb>.
Find the CUDA-GDB manual and other documentation resources online at:
    <https://docs.nvidia.com/cuda/cuda-gdb/index.html>.

For help, type "help".
Type "apropos word" to search for commands related to "word"...
Reading symbols from ./a.out...
(cuda-gdb) 

#### b. Breakpoints

- Set break

In [ ]:
!cuda-gdb -batch -ex "break simpleops_m_increment_" -ex "run" -ex "quit" ./a.out
#break simpleops_m_increment_

- Start break

In [ ]:
!cuda-gdb -batch -ex "set cuda break_on_launch application" -ex "run" -ex "quit" ./a.out
#set cuda break_on_launch application

- Run program

In [ ]:
!cuda-gdb -batch -ex "break increment" -ex "run" -ex "quit" ./a.out
#run

### c. Focus

- Help

In [ ]:
!cuda-gdb -batch -ex "break increment" -ex "run" -ex "help cuda" -ex "quit" ./a.out
#help cuda

- Focus on blocks and others where breakpoint happpened

In [ ]:
!cuda-gdb -batch -ex "break increment" -ex "run" -ex "cuda block" -ex "cuda thread" -ex "cuda warp" -ex "cuda lane" -ex "quit" ./a.out
#cuda block thread warp lane

- Change focus

In [ ]:
!cuda-gdb -batch -ex "break increment" -ex "run" -ex "cuda block 1" -ex "quit" ./a.out
#cuda block 1

#### d. Activity: Depending on focus, which depends on break

- Help

In [ ]:
!cuda-gdb -batch -ex "break increment" -ex "run" -ex "help info cuda" -ex "quit" ./a.out
#help info cuda

- Info on devices

In [ ]:
!cuda-gdb -batch -ex "break increment" -ex "run" -ex "info cuda devices" -ex "quit" ./a.out
#info cuda devices

- Info on multprocessors

In [ ]:
!cuda-gdb -batch -ex "break increment" -ex "run" -ex "info cuda sms" -ex "quit" ./a.out
#info cuda sms

- Info on warps

In [ ]:
!cuda-gdb -batch -ex "break increment" -ex "run" -ex "info cuda warps" -ex "quit" ./a.out
#info cuda warps

- Info on lanes

In [ ]:
!cuda-gdb -batch -ex "break increment" -ex "run" -ex "info cuda lanes" -ex "quit" ./a.out
#info cuda lanes

- Info on kernels, blocks and threads

In [ ]:
!cuda-gdb -batch -ex "break increment" -ex "run" -ex "info cuda kernels" -ex "quit" ./a.out
!cuda-gdb -batch -ex "break increment" -ex "run" -ex "info cuda blocks" -ex "quit" ./a.out
!cuda-gdb -batch -ex "break increment" -ex "run" -ex "info cuda threads" -ex "quit" ./a.out
#info cuda kernels
#info cuda blocks
#info cuda threads

- Colescing off

In [ ]:
!cuda-gdb -batch -ex "break increment" -ex "run" -ex "set cuda coalescing off" -ex "info cuda blocks" -ex "quit" ./a.out
#set cuda coalescing off
#info cuda blocks

#### e. Stepping

Step go into funct calls, next skips it.

- Part 1

In [ ]:
!cuda-gdb -batch -ex "break increment" -ex "run" -ex "set cuda coalescing on" -ex "info cuda threads" -ex "step" -ex "info cuda threads" -ex "quit" ./a.out
#set cuda coalescing on
#info cuda threads
#step
#info cuda threads

- Part 2

In [ ]:
!cuda-gdb -batch -ex "break increment" -ex "run" -ex "continue" -ex "quit" ./a.out
#break
#continue

#### f. Program state

- Example

In [ ]:
!cuda-gdb -batch -ex "break increment" -ex "run" -ex "print i" -ex "quit" ./a.out
!cuda-gdb -batch -ex "break increment" -ex "run" -ex "print n" -ex "quit" ./a.out

#print i
#print n

- Last block

In [ ]:
!cuda-gdb -batch -ex "break increment" -ex "run" -ex "cuda block 5 thread 0" -ex "set cuda coalescing off" -ex "info cuda lanes" -ex "quit" ./a.out
#cuda block 5 thread 0
#set cuda coalescing off
#info cuda lanes

- Step again

In [ ]:
!cuda-gdb -batch -ex "break increment" -ex "run" -ex "print a(i)" -ex "step" -ex "print a(i)" -ex "quit" ./a.out
#print a(i)
#step
#print a(i)

#### g. Sanitizer

In [ ]:
!cuda-gdb -batch -ex "break increment" -ex "run" -ex "set cuda memcheck on" -ex "quit" ./a.out
!nvfortran assertions/4_sanitizer.cuf
!compute-sanitizer --language fortran ./a.out